In [ ]:
from pathlib import Path
import os
import sys

# Make paths and `src` imports work whether Jupyter starts in the repository root
# or in the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = next(
        (parent for parent in PROJECT_ROOT.parents if (parent / "src").is_dir()),
        None,
    )
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root containing src/")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from PIL import Image
import numpy as np
from geopy.distance import geodesic
import pandas as pd
import matplotlib.pyplot as plt
import random
random.seed(7723)

# Constructing sampled sequences of 22 images from larger datasets
This notbooks samples sub-sequences of twenty-two images from the full Mapillary sequence datasets. To ensure the sequences remain fair and consistent, we will aim to have a broadly similar distance between sampled images. This distance will be the average distance between images in the smallest dataset, which contains a full 22 images (and so will be used in its entirety, unsampled). 

In [ ]:
# step one. Measure avg distance in smallest dataset

small_seq_df = pd.read_csv('downloaded_images/from_sequence/Glasgow/3HYULmaTgG8jsDdSW1xKnc/metadata.csv')

## check that the sequence is in order
plt.plot(small_seq_df.lon, small_seq_df.lat)

In [ ]:
## looks good. Lets check the average distance ebtween points

def add_distance_col(df):
    delta = []
    prev_coord = None
    for i, row in df.iterrows():
        curr_coord = (row.lat, row.lon)
        if prev_coord:
            d = geodesic(curr_coord, prev_coord)
            delta.append(d.m)
        else:
            delta.append(np.nan)
            
        prev_coord = curr_coord

    df['distance'] = delta
    return df

seq_1_df = add_distance_col(small_seq_df)
seq_1_df   

In [ ]:
# get mean distance
mean_d = np.nanmean(seq_1_df.distance)
mean_d

We'll use a target distance of 23 meters between images. Lets see how this will apply to our other datasets.

In [ ]:
id_to_df = {"3HYULmaTgG8jsDdSW1xKnc":seq_1_df}
sequence_ids = ['3HYULmaTgG8jsDdSW1xKnc',
                '6yPCl23wumovTtV5giQN8L', 
                'eyygIIVTe_eevi-HprCEow', 
                'HEoA5tNxsvPYQ1h0k46m2D',
                'jEb1lhO6caSX20KHiT8zJw',
                'PYbB9RfwVA6rS2XxTJo41L']
                
for seq_id in sequence_ids:
    seq_df = pd.read_csv(f'downloaded_images/from_sequence/Glasgow/{seq_id}/metadata.csv')
    seq_df = add_distance_col(seq_df)
    id_to_df[seq_id] = seq_df

In [ ]:
for seq_id, df in id_to_df.items():
    mean_dist = np.nanmean(df.distance)
    print(f'Sequence {seq_id} - mean distance: {mean_dist:.2f}m')
    print(f'Sequence {seq_id} - Total images: {len(df)}')

### Sampling 22 images from each image
The function below was written by ChatGPT. It uses dynamic programming to select a subsequence of 22 images from a larger list which minimises the difference between the mean distance between images and the target distance. The provided images are ordered but not necesarilly sequential - i.e. some images might be skipped. 

In [ ]:
import numpy as np
from functools import lru_cache
from tqdm import tqdm

def radnom_subset_search(image_ids, dists, target_avg=23.0, subset_size=22, n_samples=5000, max_jump=30, max_iters=50):
    N = len(image_ids)
    dists = np.nan_to_num(dists, nan=0.0)
    cum_dists = np.cumsum(dists)

    def path_distance(indices):
        return sum(cum_dists[indices[i]] - cum_dists[indices[i-1]] for i in range(1, len(indices)))

    def mean_error(indices):
        total = path_distance(indices)
        return abs((total / (len(indices) - 1)) - target_avg)

    best_error = float('inf')
    best_subset = None

    for _ in range(n_samples):
        # Step 1: initial random path with jump constraint
        max_possible_jump = max(1, (N - 1) // (subset_size - 1))
        jump = min(max_jump, max_possible_jump)

        start_upper = N - (subset_size - 1) * jump - 1
        if start_upper <= 0:
            continue
        idxs = [random.randint(0, start_upper)]
        for _ in range(1, subset_size):
            lower = idxs[-1] + 1
            upper = min(N - (subset_size - len(idxs)), idxs[-1] + jump)
            if lower >= upper:
                break
            idxs.append(random.randint(lower, upper))
        if len(idxs) != subset_size:
            continue

        # Step 2: greedy local refinement
        current = idxs.copy()
        for _ in range(max_iters):
            improved = False
            for i in range(1, subset_size - 1):
                before = current[i]
                candidates = range(current[i - 1] + 1, current[i + 1])
                best_i_error = mean_error(current)
                for c in candidates:
                    test = current.copy()
                    test[i] = c
                    test = sorted(set(test))
                    if len(test) == subset_size:
                        err = mean_error(test)
                        if err < best_i_error:
                            current = test
                            best_i_error = err
                            improved = True
            if not improved:
                break

        final_error = mean_error(current)
        if final_error < best_error:
            best_error = final_error
            best_subset = current

    if best_subset is None:
        return None

    total = path_distance(best_subset)
    return {
        "selected_indices": best_subset,
        "selected_ids": [image_ids[i] for i in best_subset],
        "mean_distance": total / (subset_size - 1),
        "total_distance": total,
        "error_from_target": best_error
    }
    
df = id_to_df['eyygIIVTe_eevi-HprCEow']
random_subset_search(df.image_id, df.distance)  

In [ ]:
images = sample_images(id_to_df['HEoA5tNxsvPYQ1h0k46m2D'])
len(images)

In [ ]:
id_to_df['HEoA5tNxsvPYQ1h0k46m2D']